# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by @id
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- Record Set @id: {record_set.id}, name: {getattr(record_set, 'name', '[no name]')}")
    # Print fields in each record set
    print("  Fields/columns (by @id):")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', '[no name]')}, dataType: {getattr(field, 'data_type', '[unknown]')}")
    for column in getattr(record_set, 'columns', []):
        print(f"    - Column @id: {column.id}, name: {getattr(column, 'name', '[no name]')}, dataType: {getattr(column, 'data_type', '[unknown]')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# First, collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set into a DataFrame, using @id reference
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns found: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print("No records found.")
print("Done loading all record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and select a numeric field for demonstration.
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # Use the first one, update as needed
    df = dataframes.get(main_record_set_id)
    print(f"Working with record set: {main_record_set_id}")
    
    # Identify numeric columns
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist() if df is not None else []
    print("Numeric columns in selected record set:", numeric_columns)
    
    # Pick first numeric column for example
    if numeric_columns:
        numeric_field = numeric_columns[0]
        print(f"Using numeric field: {numeric_field}")

        threshold = df[numeric_field].mean()  # Example threshold: mean value

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a string/categorical field if available
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found to group by.")
    else:
        print("No numeric fields found in the chosen record set. Choose another record set or check your data.")
else:
    print("No record sets loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field, if it exists
if record_set_ids and numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If a grouping field is found, plot group means
    if group_candidates:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded from a FAIR² Croissant schema.
- Record sets and fields were explored using their `@id` attributes, ensuring traceability to the schema.
- We loaded tabular data into DataFrames, examined numeric fields, performed basic filtering, normalization, and grouping for EDA.
- Distributions and group means were visualized to facilitate further insights into mental health survey data from Kilifi County, Kenya.

For a more in-depth analysis, refer to the schema and documentation for the full description of surveyed variables and their context.